# Loan Default Baseline Model

This notebook loads the `Loan_Default.csv` dataset, preprocesses it, trains a baseline classifier, and evaluates the model.

Run the cells in order.

In [1]:
# Install missing packages in a terminal if needed:
# py -m pip install pandas scikit-learn numpy gradio

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, roc_auc_score

print('Libraries imported successfully.')


Libraries imported successfully.


In [2]:
path = r'C:\ML_files\Project\Loan_Default.csv'
df = pd.read_csv(path, low_memory=False)

print('Dataset shape:', df.shape)
print('Columns:', list(df.columns))
print('Head:')
display(df.head())
print('Target distribution:')
print(df['Status'].value_counts(dropna=False))


Dataset shape: (148670, 34)
Columns: ['ID', 'year', 'loan_limit', 'Gender', 'approv_in_adv', 'loan_type', 'loan_purpose', 'Credit_Worthiness', 'open_credit', 'business_or_commercial', 'loan_amount', 'rate_of_interest', 'Interest_rate_spread', 'Upfront_charges', 'term', 'Neg_ammortization', 'interest_only', 'lump_sum_payment', 'property_value', 'construction_type', 'occupancy_type', 'Secured_by', 'total_units', 'income', 'credit_type', 'Credit_Score', 'co-applicant_credit_type', 'age', 'submission_of_application', 'LTV', 'Region', 'Security_Type', 'Status', 'dtir1']
Head:


,ID,year,loan_limit,Gender,approv_in_adv,loan_type,loan_purpose,Credit_Worthiness,open_credit,business_or_commercial,...,credit_type,Credit_Score,co-applicant_credit_type,age,submission_of_application,LTV,Region,Security_Type,Status,dtir1
0,24890,2019,cf,Sex Not Available,nopre,type1,p1,l1,nopc,nob/c,...,EXP,758,CIB,25-34,to_inst,98.728814,south,direct,1,45.0
1,24891,2019,cf,Male,nopre,type2,p1,l1,nopc,b/c,...,EQUI,552,EXP,55-64,to_inst,NaN,North,direct,1,NaN
2,24892,2019,cf,Male,pre,type1,p1,l1,nopc,nob/c,...,EXP,834,CIB,35-44,to_inst,80.019685,south,direct,0,46.0
3,24893,2019,cf,Male,nopre,type1,p4,l1,nopc,nob/c,...,EXP,587,CIB,45-54,not_inst,69.376900,North,direct,0,42.0
4,24894,2019,cf,Joint,pre,type1,p1,l1,nopc,nob/c,...,CRIF,602,EXP,25-34,not_inst,91.886544,North,direct,0,39.0


Target distribution:
Status
0    112031
1     36639
Name: count, dtype: int64


In [3]:
target = 'Status'

# Remove rows with no target value
df = df.dropna(subset=[target]).copy()

# Exclude the record ID if present
X = df.drop(columns=[target, 'ID'], errors='ignore')
y = df[target]

print('Missing values before preprocessing:')
print(X.isna().sum().sort_values(ascending=False).head(10).to_string())

numeric_cols = X.select_dtypes(include='number').columns.tolist()
categorical_cols = X.select_dtypes(exclude='number').columns.tolist()
print(f'Numeric columns: {len(numeric_cols)}')
print(f'Categorical columns: {len(categorical_cols)}')


Missing values before preprocessing:
Upfront_charges         39642
Interest_rate_spread    36639
rate_of_interest        36439
dtir1                   24121
property_value          15098
LTV                     15098
income                   9150
loan_limit               3344
approv_in_adv             908
age                       200
Numeric columns: 11
Categorical columns: 21


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

preprocess = ColumnTransformer(
    transformers=[
        (
            'num',
            Pipeline(steps=[
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())
            ]),
            numeric_cols
        ),
        (
            'cat',
            Pipeline(steps=[
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder', OneHotEncoder(handle_unknown='ignore', min_frequency=10))
            ]),
            categorical_cols
        )
    ]
)

model = Pipeline(steps=[
    ('preprocess', preprocess),
    ('classifier', LogisticRegression(max_iter=500, class_weight='balanced', solver='lbfgs'))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print('Train/test shapes:', X_train.shape, X_test.shape)
print('Accuracy:', round(accuracy_score(y_test, y_pred), 4))
print('Balanced accuracy:', round(balanced_accuracy_score(y_test, y_pred), 4))
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred))
print('Classification report:')
print(classification_report(y_test, y_pred, zero_division=0))

# ROC AUC is only valid for binary classification
if len(model.named_steps['classifier'].classes_) == 2:
    y_proba = model.predict_proba(X_test)[:, 1]
    print('ROC AUC:', round(roc_auc_score(y_test, y_proba), 4))


Train/test shapes: (118936, 32) (29734, 32)
Accuracy: 0.8388
Balanced accuracy: 0.7958
Confusion matrix:
[[19732  2674]
 [ 2118  5210]]
Classification report:
              precision    recall  f1-score   support

           0       0.90      0.88      0.89     22406
           1       0.66      0.71      0.68      7328

    accuracy                           0.84     29734
   macro avg       0.78      0.80      0.79     29734
weighted avg       0.84      0.84      0.84     29734

ROC AUC: 0.8675


## Results from the baseline model

The run on the provided dataset produced the following metrics on the held-out test set:
- Accuracy: 0.8388
- Balanced accuracy: 0.7958
- ROC AUC: 0.8675

This confirms the model is working on the dataset and is ready for further improvements such as feature engineering, hyperparameter tuning, or a Gradio app.